In [2]:
pip install gensim

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# import necessary libraries
import pandas as pd
import gensim 
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string
import nltk
import re


In [5]:
df = pd.read_csv("https://github.com/Ankit152/IMDB-sentiment-analysis/raw/refs/heads/master/IMDB-Dataset.csv")

In [6]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [7]:
from sklearn.model_selection import train_test_split

In [8]:
X_train,X_test,Y_train,Y_test = train_test_split(
    df['review'],
    df['sentiment'],
    test_size=0.5,
    random_state=42,
    stratify=df['sentiment'] # to make sure balance of split in data 12.5 positive and 12.5 negative
)

In [9]:
print('Training set:',X_train.shape)
print(Y_train.value_counts())

print("\nTesting set:",X_test.shape)
print(Y_test.value_counts())

Training set: (25000,)
sentiment
positive    12500
negative    12500
Name: count, dtype: int64

Testing set: (25000,)
sentiment
negative    12500
positive    12500
Name: count, dtype: int64


In [10]:
nltk.download('punkt') # needed for tokenozation
nltk.download('stopwords') #    needed for removation of stopwords 

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [11]:
nltk.download('punkt_tab') 
nltk.download('stopwords')

# preprocess function 
def preprocess(text):
    text = re.sub(r'<[^>]+>', ' ', text)  # Remove HTML tags
    # lowercase , remove punctuation and tokenize
    text = text.lower()
    text = ''.join([char for char in text if char not in string.punctuation])
    words = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    words = [word for word in words if word not in stop_words]
    return words

# Preprocess training and testing reviews separately
processed_train = X_train.apply(preprocess)
processed_test = X_test.apply(preprocess)


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [12]:
processed_train

3055     [trios, vignettes, insightful, quite, enjoyabl...
47326    [never, heard, one, owner, local, dvd, rental,...
48869    [excellent, hard, find, trippy, world, war, sp...
48623    [huge, fan, big, loud, trashy, completely, stu...
39657    [easily, lucio, fulcis, respected, film, dont,...
                               ...                        
21552    [recent, years, seen, number, biopics, famous,...
10610    [may, say, johnson, elvis, work, possible, see...
20104    [title, refers, questionable, poker, hand, six...
35597    [far, pathetic, movie, indian, cinema, cinema,...
27491    [movie, forever, left, impression, watched, fr...
Name: review, Length: 25000, dtype: object

In [13]:
processed_test

16346    [sixyearold, boy, whos, dinosaurs, love, movie...
42137    [voted, 1, film, noticed, 75, people, voted, 1...
21867    [arent, talking, animals, big, lavish, song, p...
12577    [ever, think, like, watching, horror, movie, g...
25444    [expecting, love, moviefilm, noir, serial, kil...
                               ...                        
32628    [read, comments, site, feel, compelled, write,...
8352     [people, expect, jurassic, park, 3, full, surp...
49159    [fan, probably, enjoy, dont, know, misty, mund...
38453    [ever, since, played, goon, lone, wolf, mcquad...
47518    [found, movie, nearly, impossible, watch, supe...
Name: review, Length: 25000, dtype: object

In [14]:
# train word2vec model on the processed text data 
model = Word2Vec(sentences=processed_train,vector_size=100,window=5,min_count=1,workers=4)

In [15]:
model.wv.vector_size

100

In [16]:
model.wv['wonderful']

array([-1.7315596 , -1.3352052 , -0.38852733, -1.5182948 , -0.06891279,
       -3.3512263 , -0.39556965,  1.6702801 , -0.18448035,  0.28200498,
       -0.27037448, -0.40674093, -2.2779262 ,  1.1630257 , -0.64165324,
        0.38957718, -0.08617048, -0.02121843, -1.1301757 , -0.60148627,
        1.7202768 ,  0.5871589 ,  1.5787268 ,  1.7586375 ,  0.3883797 ,
        1.8934618 , -1.0977014 , -0.4875776 , -1.1731733 , -1.3033153 ,
       -0.04626658,  0.09080425,  1.8170087 ,  0.72451246, -0.28464618,
        0.9799534 ,  0.07884976, -2.1141376 , -1.6371441 ,  0.53696805,
        0.5964545 ,  0.03505341, -0.7179217 , -1.1727285 ,  1.9366455 ,
        1.0308594 , -1.1714578 , -0.4919202 ,  1.7857151 , -0.96359575,
       -0.8821081 , -2.442683  ,  0.41220146,  1.1026812 , -0.71868294,
        0.0798724 , -2.065604  , -2.0040057 ,  0.5796638 , -0.35933492,
        0.3446793 ,  1.0415773 , -0.34949958,  0.45803228, -0.69335854,
        1.0898645 , -0.5935104 ,  0.21044622,  0.83597666,  0.00

In [17]:
model.wv['going']

array([ 0.910929  ,  0.02394794,  0.17350629, -0.01417368,  0.15227409,
       -0.86588717,  0.07203917,  1.5040712 , -3.2610657 ,  0.49120727,
       -1.439463  , -1.2212437 ,  0.0054912 , -1.1754955 ,  1.808875  ,
       -0.42188022,  0.35236445, -0.0986762 ,  0.9663365 , -3.2429123 ,
        1.1418097 ,  0.07737677, -1.2867494 ,  1.5410426 , -0.45990765,
       -1.6082575 , -2.660917  , -2.1659052 ,  0.48282972,  0.01589872,
       -0.1838612 ,  1.0151827 ,  1.8738724 , -1.9876164 , -0.96661556,
        0.9987846 , -1.3043797 , -0.6443283 ,  0.709417  , -1.126088  ,
        0.5755276 ,  0.51896805,  1.0155159 ,  1.9949514 ,  1.8765165 ,
       -0.49633962, -0.18981223,  0.13170265, -0.20198452, -0.8485981 ,
       -2.043402  , -0.8033076 , -0.6947173 , -0.684389  ,  0.50909716,
        0.8644599 ,  1.135126  ,  0.73265404,  0.02438079, -1.3491142 ,
       -0.08123472,  0.898995  ,  1.0207402 ,  0.8039491 , -2.4381166 ,
       -0.26479068,  0.00375438,  0.89202994, -2.6658573 , -2.09

In [18]:
len(model.wv.index_to_key)

113140

In [19]:
sp = model.wv['speak'] # it is the vector representation of speak word

In [20]:
fc = model.wv['force']

In [21]:
cosine_similarity([sp,fc])

array([[1.       , 0.3405428],
       [0.3405428, 1.0000001]], dtype=float32)

In [22]:
model.wv.index_to_key

['movie',
 'film',
 'one',
 'like',
 'good',
 'even',
 'would',
 'time',
 'really',
 'story',
 'see',
 'much',
 'well',
 'bad',
 'get',
 'also',
 'great',
 'people',
 'first',
 'dont',
 'films',
 'movies',
 'made',
 'could',
 'way',
 'make',
 'characters',
 'think',
 'watch',
 'seen',
 'character',
 'many',
 'two',
 'never',
 'love',
 'acting',
 'plot',
 'best',
 'know',
 'little',
 'show',
 'life',
 'ever',
 'better',
 'still',
 'scene',
 'say',
 'end',
 'man',
 'scenes',
 'something',
 'go',
 'im',
 'back',
 'real',
 'thing',
 'watching',
 'actors',
 'doesnt',
 'didnt',
 'years',
 'though',
 'funny',
 'actually',
 'another',
 'find',
 'makes',
 'work',
 'look',
 'nothing',
 'going',
 'lot',
 'new',
 'every',
 'old',
 'us',
 'part',
 'cant',
 'director',
 'thats',
 'quite',
 'cast',
 'things',
 'got',
 'pretty',
 'want',
 'enough',
 'around',
 'world',
 'seems',
 'however',
 'young',
 'take',
 'fact',
 'horror',
 'thought',
 'ive',
 'give',
 'big',
 'may',
 'long',
 'always',
 'withou

In [23]:
print("Training reviews:", len(processed_train))
print("Testing reviews:", len(processed_test))
print("Word2Vec vocabulary:", len(model.wv))
print("Vector size:", model.vector_size)

Training reviews: 25000
Testing reviews: 25000
Word2Vec vocabulary: 113140
Vector size: 100


In [24]:
review = processed_train.iloc[0]

# converting each word into its word2vec vector
vectors = [model.wv[word] for word in review if word in model.wv]
print("Number of words:",len(review))
print("Number of vectors:",len(vectors))
print("Shape of one word vector:",vectors[0].shape)

Number of words: 61
Number of vectors: 61
Shape of one word vector: (100,)


In [25]:
len(processed_train.iloc[1])

142

In [26]:
# Checking the number of words in each training review
review_lengths = processed_train.apply(len)

print("Shortest review:", review_lengths.min())
print("Longest review:", review_lengths.max())
print("Average review length:", review_lengths.mean())
print("95th percentile:", review_lengths.quantile(0.95))

Shortest review: 3
Longest review: 1437
Average review length: 120.00644
95th percentile: 309.0


In [27]:
# converting each review into a sequnce of word2vec vectors
X_train_vectors = [[model.wv[word] for word in review if word in model.wv]
for review in processed_train]



In [28]:
X_test_vectors = [
    [model.wv[word] for word in review if word in model.wv]
    for review in processed_test
]

In [29]:
print("Training reviews:", len(X_train_vectors))
print("Testing reviews:", len(X_test_vectors))

print("First training review — words:", len(X_train_vectors[0]))
print("First training review — vector shape:", X_train_vectors[0][0].shape)

Training reviews: 25000
Testing reviews: 25000
First training review — words: 61
First training review — vector shape: (100,)


In [30]:
# padding and truncation

import numpy as np

max_length = 309
vector_size = model.vector_size

X_train_padded = np.zeros(
    (len(X_train_vectors), max_length, vector_size),
    dtype=np.float32
)

for i, review_vectors in enumerate(X_train_vectors):
    review_vectors = np.asarray(review_vectors, dtype=np.float32)
    length = min(len(review_vectors), max_length)
    X_train_padded[i, :length] = review_vectors[:length]

    

In [31]:
print("Training array shape:", X_train_padded.shape)
print("Training array data type:", X_train_padded.dtype)

Training array shape: (25000, 309, 100)
Training array data type: float32


In [32]:
from sklearn.preprocessing import LabelEncoder

In [33]:
le = LabelEncoder()

y_train_encoded = le.fit_transform(Y_train)
y_test_encoded = le.transform(Y_test)

print("Classes:", le.classes_)
print("Training labels shape:", y_train_encoded.shape)
print("First 10 labels:", y_train_encoded[:10])

Classes: ['negative' 'positive']
Training labels shape: (25000,)
First 10 labels: [1 0 1 1 1 0 1 1 0 1]


In [34]:

print("Original review:")
print(X_test.iloc[35])

print("\nProcessed tokens:")
print(processed_test.iloc[35])

print("\nNumber of processed tokens:",
      len(processed_test.iloc[35]))

print("\nNumber of vectors:",
      len(X_test_vectors[35]))

Original review:
Shame Shame Shame on UA/DW for what you do! <br /><br />I was appalled. <br /><br />Do NOT take kids to see this movie. The humor is totally inappropriate for children - plus they'll be bored and disappointed. Certainly *we all* have read Theo's wonderful children book and certainly we have expectations...but this is pure trash. Dr. Seuss would be ashamed and certainly would've never given his "thumbs up" at such a dastardly attempt to capitalize on a classic.<br /><br />What a pity. <br /><br />Spend your money on the book. If you own a copy, then buy the book and donate it to a Toys for Tots program. This movie is NOT worth a "free" ticket viewing.<br /><br />Stick with the book. The tv cartoon version works well if you want a visual portrayal - save your money...seriously. SAVE your money - it will be on cable by saint patty's day.<br /><br />Shame shame shame on what they do!!

Processed tokens:
['shame', 'shame', 'shame', 'uadw', 'appalled', 'take', 'kids', 'see',

In [35]:
import tensorflow as tf

In [36]:
print("TensorFlow version:", tf.__version__)
print("Word2Vec vocabulary size:", len(model.wv))
print("Word2Vec vector size:", model.vector_size)
print("Maximum sequence length:", max_length)

TensorFlow version: 2.21.0
Word2Vec vocabulary size: 113140
Word2Vec vector size: 100
Maximum sequence length: 309


In [37]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input,LSTM,Dense,Dropout

In [38]:
model_lstm = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(309,100)),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(32,activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1,activation='sigmoid')
])

In [39]:
model_lstm.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        42,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,353 (173.25 KB)

 Trainable params: 44,353 (173.25 KB)

 Non-trainable params: 0 (0.00 B)

In [40]:
model_lstm.compile(loss='binary_crossentropy',
                   optimizer='adam',
                   metrics=['accuracy'])

In [41]:
print("X_train_padded shape:", X_train_padded.shape)
print("X_train_padded dtype:", X_train_padded.dtype)

print("y_train_encoded shape:", y_train_encoded.shape)
print("y_train_encoded dtype:", y_train_encoded.dtype)

print("X_train_padded memory (GB):",
      X_train_padded.nbytes / (1024**3))

print("Label classes:", np.unique(y_train_encoded))

X_train_padded shape: (25000, 309, 100)
X_train_padded dtype: float32
y_train_encoded shape: (25000,)
y_train_encoded dtype: int64
X_train_padded memory (GB): 2.8777867555618286
Label classes: [0 1]


In [42]:
model_lstm.fit(X_train_padded,y_train_encoded,epochs=10,batch_size=16,validation_split=0.1,verbose=1)

Epoch 1/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 158s 66ms/step - accuracy: 0.5377 - loss: 0.6833 - val_accuracy: 0.5040 - val_loss: 0.6926
Epoch 2/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 65s 45ms/step - accuracy: 0.5124 - loss: 0.6897 - val_accuracy: 0.5192 - val_loss: 0.6864
Epoch 3/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 65s 46ms/step - accuracy: 0.5668 - loss: 0.6633 - val_accuracy: 0.7308 - val_loss: 0.5753
Epoch 4/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 69s 49ms/step - accuracy: 0.6660 - loss: 0.6091 - val_accuracy: 0.5880 - val_loss: 0.6427
Epoch 5/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 66s 47ms/step - accuracy: 0.6866 - loss: 0.6055 - val_accuracy: 0.5396 - val_loss: 0.6843
Epoch 6/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 65s 46ms/step - accuracy: 0.6810 - loss: 0.5979 - val_accuracy: 0.5148 - val_loss: 0.6872
Epoch 7/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 65s 46ms/step - accuracy: 0.5163 - loss: 0.6834 - val_accuracy: 0.5124 - val_loss: 0.6851
Epoch 8/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 64s 46ms/step - accuracy: 0.6220 

In [43]:
X_test_padded = np.zeros(
    (len(X_test_vectors), max_length, vector_size),
    dtype=np.float32
)

for i, review_vectors in enumerate(X_test_vectors):
    length = min(len(review_vectors), max_length)

    if length > 0:
        X_test_padded[i, :length] = np.asarray(
            review_vectors[:length],
            dtype=np.float32
        )

print(X_test_padded.shape)

(25000, 309, 100)


In [44]:
y_pred_prob = model_lstm.predict(X_test_padded)

y_pred = (y_pred_prob >= 0.5).astype(int)

print("Predictions shape:", y_pred.shape)
print("First 10 predictions:", y_pred[:10].flatten())
print("First 10 actual labels:", y_test_encoded[:10])

782/782 ━━━━━━━━━━━━━━━━━━━━ 64s 29ms/step
Predictions shape: (25000, 1)
First 10 predictions: [0 0 0 0 0 1 0 1 0 0]
First 10 actual labels: [0 0 1 0 0 1 0 1 0 0]


In [45]:
from sklearn.metrics import accuracy_score

test_accuracy = accuracy_score(y_test_encoded,y_pred)

print("Test Accuracy:",test_accuracy)
print("Test Accuracy (%):", test_accuracy * 100)

Test Accuracy: 0.8606
Test Accuracy (%): 86.06


In [46]:
# confusion matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_encoded, y_pred)

print(cm)

[[10914  1586]
 [ 1899 10601]]


In [47]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test_encoded,
    y_pred,
    target_names=['Negative','Positive']
)
)

              precision    recall  f1-score   support

    Negative       0.85      0.87      0.86     12500
    Positive       0.87      0.85      0.86     12500

    accuracy                           0.86     25000
   macro avg       0.86      0.86      0.86     25000
weighted avg       0.86      0.86      0.86     25000



In [49]:
from gensim.models import Word2Vec

model = Word2Vec.load("word2vec.model")

model.wv.most_similar("cold")

[('animosity', 0.7845356464385986),
 ('lurking', 0.7671880722045898),
 ('anti', 0.761338472366333),
 ('oppression', 0.756351113319397),
 ('harbors', 0.737354576587677),
 ('threat', 0.7360300421714783),
 ('inhuman', 0.7329788208007812),
 ('arid', 0.7309819459915161),
 ('visions', 0.728202223777771),
 ('devastation', 0.7277215719223022)]